# Minerva Evals

## Results folders

In [28]:
from minerva_eval import load_all_results, get_run_summary, metrics_heatmap, EvalResult, Hit, gold_ranks
import pandas as pd
import json
from pathlib import Path
import re

all_results = load_all_results()

collections = {r.collection for r in all_results}
print(f"{len(all_results)} rows from {len(collections)} collections: {collections}")

972 rows from 2 collections: {'wikipedia-qwen2-5', 'wikipedia-nollm'}


In [29]:
import ipywidgets as widgets

labels = sorted({r.label for r in all_results})
picker = widgets.SelectMultiple(options=labels, value=tuple(labels),
                                description="runs", rows=(len(labels) + 2), layout=widgets.Layout(width="700px"))
display(picker)


SelectMultiple(description='runs', index=(0, 1), layout=Layout(width='700px'), options=('78 queries (old)', 'b…

## Run summary

One row per run folder: collection, timestamp, sweep matrix, query count.

In [35]:
run_summary = get_run_summary()
run_summary

,folder,collection,timestamp,sweep,queries,rows
0,2026-08-21T07-05-38Z_wikipedia-v1,wikipedia-qwen2-5,2026-08-21T07:05:38Z,"top_k=[50] alpha=[0.3, 0.5, 0.7]",84,252
1,2026-08-21T07-08-27Z_wikipedia-v1,wikipedia-nollm,2026-08-21T07:08:27Z,"top_k=[50] alpha=[0.3, 0.5, 0.7]",84,252
2,2026-08-20T12-25-48Z_wikipedia-v1,wikipedia-nollm,2026-08-20T12:25:48Z,"top_k=[50] alpha=[0.3, 0.5, 0.7]",78,234
3,2026-08-20T12-27-20Z_wikipedia-v1,wikipedia-qwen2-5,2026-08-20T12:27:20Z,"top_k=[50] alpha=[0.3, 0.5, 0.7]",78,234


## Metrics view

Mean over queries, grouped by `(collection, hybrid_alpha)`. Reproduces the
headline table from the sweep-findings spec; styled as a heatmap.

In [36]:
chosen = [r for r in all_results if r.label in picker.value]
metrics_heatmap(chosen, group_cols=("label", "collection", "alpha"))

## Wrong queries finder

Pick failures for R@10 for any $\alpha$ as they may potentially identify wrong gold sources

In [37]:
from collections import defaultdict

def persistent_failures(collection: str, metric: str, threshold: float = 1.0):
    # aggregates all_results by query id
    by_query = defaultdict(list)
    for r in all_results:
        if r.collection == collection:
            by_query[r.query_id].append(r)
    return [rs for rs in by_query.values()
            if all(getattr(r.metrics, metric) < threshold for r in rs)]

metric = "recall_at_10"
failures = persistent_failures("wikipedia-nollm", metric)

print(f"{len(failures)} suspect queries")
pd.DataFrame([
    { 
     "query_id": rs[0].query_id, 
     "query": rs[0].query, 
     **{f"α={r.cell.hybrid_alpha}": getattr(r.metrics, metric) for r in sorted(rs, key=lambda r: r.cell.hybrid_alpha)},
    }
    for rs in failures
])

0 suspect queries


""


## Audit view

For each failing query: the gold article(s) with a corpus snippet, plus the
deduplicated top-10 retrieved articles (rank + GOLD marker). Lets each miss be
classified as a real retrieval failure or a mislabeled gold.

The corpus path is read from `config.local.json` (gitignored, machine-specific).
Copy `config.local.json.example` and set `corpus_dir`.

In [33]:
config = json.loads(Path("config.local.json").read_text())
CORPUS_DIR = Path(config["corpus_dir"])

def read_snippet(source_id: str, max_chars: int = 300) -> str:
    text = (CORPUS_DIR / source_id).read_text()
    body = re.sub(r"^---\n.*?\n---\n", "", text, count=1, flags=re.DOTALL)  # drop frontmatter
    paras = [s for p in body.split("\n\n") if (s := p.strip()) and not s.startswith("#")]
    snippet = paras[0] if paras else ""
    return snippet[:max_chars] + ("…" if len(snippet) > max_chars else "")

def top_articles(hits: list[Hit], k: int = 10) -> list[Hit]:
    seen, out = set(), []
    for h in sorted(hits, key=lambda h: h.rank):
        if h.source_id not in seen:
            seen.add(h.source_id)
            out.append(h)
        if len(out) == k:
            break
    return out

from minerva_eval import best_rank_by_source

def audit(groups: list[list[EvalResult]]) -> None:
    for rs in groups:
        rs = sorted(rs, key=lambda r: r.cell.hybrid_alpha)
        rep = rs[1]  # representative for the top-10 dump
        print(f"=== {rep.query_id} ===")
        print(f"query: {rep.query!r}")
        for g in rep.gold_sources:
            ranks = {r.cell.hybrid_alpha: best_rank_by_source(r).get(g, "—") for r in rs}
            print(f"gold  [{g}] ranks {ranks}: {read_snippet(g)}")
        print(f"top-10 articles (α={rep.cell.hybrid_alpha}):")
        for h in top_articles(rep.hits):
            mark = "GOLD" if h.source_id in rep.gold_sources else "    "
            print(f"  {h.rank:>2} {mark} {h.source_id}  ({h.score:.4f})")
        print()

audit(failures)


## Gold rank table

Value is the gold's worst source rank in that cell. Queries with multiple golds
collapse with `max` (the worst-placed gold). Rows are sorted hardest
first (largest worst-case rank on top), so trivially easy queries (rank 1 in
every cell) sink to the bottom. Green = low rank (easy), red = high rank (hard).

In [38]:
ranks = pd.DataFrame(vars(g) for g in gold_ranks(chosen))   # was: gold_ranks(all_results)

table = ranks.pivot_table(
    index="query_id", columns=["collection", "label", "alpha"],
    values="rank", 
    aggfunc="max",          # multi-gold -> all have been found?
)

table = table.loc[table.max(axis=1).sort_values(ascending=False).index]

table.style.background_gradient(cmap="RdYlGn_r", axis=None)